In [1]:
# IMPORTATIONS DES OUTILS (SCIKIT-LEARN) ────────────────────────────────
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, confusion_matrix
from sklearn.preprocessing import StandardScaler, OneHotEncoder

import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
# setting Jedha color palette as default
pio.templates["jedha"] = go.layout.Template(
    layout_colorway=["#4B9AC7", "#4BE8E0", "#9DD4F3", "#97FBF6", "#2A7FAF", "#23B1AB", "#0E3449", "#015955"]
)
pio.templates.default = "jedha"
pio.renderers.default = "iframe" # to be replaced by "iframe" if working on JULIE
from IPython.display import display


In [2]:
df = pd.read_csv('../data/conversion_data_train.csv')
print('Set with labels (our train+test) :', df.shape)
display(df.head())


Set with labels (our train+test) : (284580, 6)


,country,age,new_user,source,total_pages_visited,converted
0,China,22,1,Direct,2,0
1,UK,21,1,Ads,3,0
2,Germany,20,0,Seo,14,1
3,US,23,1,Seo,3,0
4,US,28,1,Direct,3,0


In [3]:
df_sample = df.sample(10000)

In [4]:
print("Number of rows : {}".format(df.shape[0]))
print()

print("Display of dataset: ")
display(df.head())
print()

print("Basics statistics: ")
data_desc = df.describe(include="all")
display(data_desc)
print()

print("Percentage of missing values: ")
display(100 * df.isnull().sum() / df.shape[0])

Number of rows : 284580

Display of dataset: 


,country,age,new_user,source,total_pages_visited,converted
0,China,22,1,Direct,2,0
1,UK,21,1,Ads,3,0
2,Germany,20,0,Seo,14,1
3,US,23,1,Seo,3,0
4,US,28,1,Direct,3,0



Basics statistics: 


,country,age,new_user,source,total_pages_visited,converted
count,284580,284580.000000,284580.000000,284580,284580.000000,284580.000000
unique,4,NaN,NaN,3,NaN,NaN
top,US,NaN,NaN,Seo,NaN,NaN
freq,160124,NaN,NaN,139477,NaN,NaN
mean,NaN,30.564203,0.685452,NaN,4.873252,0.032258
std,NaN,8.266789,0.464336,NaN,3.341995,0.176685
min,NaN,17.000000,0.000000,NaN,1.000000,0.000000
25%,NaN,24.000000,0.000000,NaN,2.000000,0.000000
50%,NaN,30.000000,1.000000,NaN,4.000000,0.000000
75%,NaN,36.000000,1.000000,NaN,7.000000,0.000000



Percentage of missing values: 


country                0.0
age                    0.0
new_user               0.0
source                 0.0
total_pages_visited    0.0
converted              0.0
dtype: float64

In [5]:
# On inclut les colonnes utiles, pas juste les pages visitées !
features_list = ['country', 'age', 'new_user', 'source', 'total_pages_visited']
target_variable = 'converted'
X = df[features_list]
Y = df[target_variable]

print('Explanatory variables : ', X.columns)
print()



Explanatory variables :  Index(['country', 'age', 'new_user', 'source', 'total_pages_visited'], dtype='str')



In [6]:
print("Dividing into train and test sets...")
# UPGRADE LEAD : Ajout de stratify=Y pour gérer le déséquilibre 97/3
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, stratify=Y, random_state=0)
print("...Done.")
print()

Dividing into train and test sets...
...Done.



In [7]:
print("Encoding categorical features and standardizing numerical features...")

numeric_features = ['age', 'total_pages_visited']
categorical_features = ['country', 'new_user', 'source']

featureencoder = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_features)
    ])

# On ÉCRASE l'ancien X_train (avec les mots) par le nouveau (avec les chiffres)
X_train = featureencoder.fit_transform(X_train)

print("...Done")
print(X_train[0:5,:])


Encoding categorical features and standardizing numerical features...
...Done
[[-1.3990984  -1.15935344  0.          0.          0.          1.
   0.          1.        ]
 [ 0.29299544  0.03743241  0.          0.          1.          1.
   1.          0.        ]
 [ 2.46854467 -0.86015697  0.          0.          1.          1.
   0.          0.        ]
 [-1.64082609 -1.15935344  0.          0.          0.          0.
   0.          1.        ]
 [-0.31132378  0.03743241  0.          0.          0.          1.
   0.          1.        ]]


In [8]:
print("Train model...")
classifier = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=0) 

classifier.fit(X_train, Y_train)

print("...Done.")

Train model...
...Done.


In [9]:
print("Predictions on training set...")
Y_train_pred = classifier.predict(X_train)
print("...Done.")
print(Y_train_pred)
print()

Predictions on training set...
...Done.
[0 0 0 ... 0 1 0]



In [10]:
print("Encoding categorical features and standardizing numerical features...")
X_test = featureencoder.transform(X_test)
print("...Done")
print(X_test[0:5,:])

Encoding categorical features and standardizing numerical features...
...Done
[[ 0.41385929 -1.15935344  0.          1.          0.          1.
   0.          0.        ]
 [ 0.1721316   0.03743241  0.          1.          0.          0.
   0.          0.        ]
 [ 1.62249775 -1.15935344  0.          0.          1.          1.
   0.          0.        ]
 [ 0.53472314 -1.15935344  0.          0.          1.          1.
   1.          0.        ]
 [-0.19045994 -0.56096051  0.          0.          1.          1.
   1.          0.        ]]


In [11]:
print("Predictions on test set...")
Y_test_pred = classifier.predict(X_test)
print("...Done.")
print(Y_test_pred)
print()

Predictions on test set...
...Done.
[0 0 0 ... 0 0 0]



In [12]:
print("f1-score on train set : ", f1_score(Y_train, Y_train_pred))
print("f1-score on test set : ", f1_score(Y_test, Y_test_pred))

print("Confusion matrix on train set : ")
print(confusion_matrix(Y_train, Y_train_pred))
print()
print("Confusion matrix on test set : ")
print(confusion_matrix(Y_test, Y_test_pred))
print()

f1-score on train set :  0.5094269870609982
f1-score on test set :  0.5079552150854449
Confusion matrix on train set : 
[[207504  12816]
 [   454   6890]]

Confusion matrix on test set : 
[[51852  3228]
 [  112  1724]]



In [13]:
print("Entraînement sur toutes les données...")

X_all_processed = featureencoder.fit_transform(X)
classifier.fit(X_all_processed, Y)
print("...Done.")

Entraînement sur toutes les données...
...Done.


In [14]:
data_without_labels = pd.read_csv('../data/conversion_data_test.csv')
print('Prediction set (without labels) :', data_without_labels.shape)

features_list = ['country', 'age', 'new_user', 'source', 'total_pages_visited']
X_without_labels = data_without_labels.loc[:, features_list]

print("Données chargées avec succès !")

Prediction set (without labels) : (31620, 5)
Données chargées avec succès !


In [15]:
print("Encoding categorical features and standardizing numerical features...")
X_without_labels = featureencoder.transform(X_without_labels)
print("...Done")
print(X_without_labels[0:5,:])

Encoding categorical features and standardizing numerical features...
...Done
[[-0.31018183  3.3293794   0.          1.          0.          0.
   0.          1.        ]
 [-1.03597878  0.03792598  0.          1.          0.          1.
   1.          0.        ]
 [ 0.1736828  -1.15896618  0.          0.          0.          1.
   0.          1.        ]
 [ 0.1736828   0.33714902  0.          0.          1.          1.
   0.          0.        ]
 [-0.67308031 -0.5605201   0.          0.          0.          0.
   0.          1.        ]]


In [16]:
data = {
    'converted': classifier.predict(X_without_labels)
}

Y_predictions = pd.DataFrame(columns=['converted'],data=data)
Y_predictions.to_csv('conversion_data_test_predictions_CARELLE.csv', index=False)
print("Fichier de soumission créé !")

Fichier de soumission créé !


In [17]:
pio.renderers.default = 'vscode'

# 1. On récupère les noms bruts
raw_names = featureencoder.get_feature_names_out()

# 2. NETTOYAGE VIP : On enlève les préfixes laids de Sklearn
clean_names = [name.replace('num__', '').replace('cat__', '').replace('_', ' ').title() for name in raw_names]

# 3. Création du tableau
coefs = pd.DataFrame({
    'Critère': clean_names,
    'Coefficient': classifier.coef_[0]
}).sort_values(by='Coefficient', ascending=True) # Ascending=True pour un plus bel ordre d'affichage

# 4. Le graphique version "Lead Data Scientist"
fig = px.bar(
    coefs, 
    x='Coefficient', 
    y='Critère', 
    orientation='h',
    title="<b>Qu'est-ce qui pousse un utilisateur à s'abonner ?</b><br><sup>Poids des variables (Régression Logistique)</sup>",
    color='Coefficient',
    color_continuous_scale='RdBu',
    text_auto='.2f' # Ajoute les chiffres directement sur les barres !
)

# 5. Ajustement des marges et du design de la fenêtre
fig.update_layout(
    width=900,
    height=550,
    template='plotly_white', # Un fond blanc pur, plus élégant
    yaxis_title=None, # On enlève le titre de l'axe Y qui prend de la place pour rien
    xaxis_title="Impact sur la conversion (Log-Odds)",
    margin=dict(l=150, r=40, t=80, b=40), # On force une grosse marge à gauche (l=150)
    font=dict(size=13)
)

fig.show()

In [18]:
column_names = featureencoder.get_feature_names_out()

pio.renderers.default = 'vscode' 

column_names = featureencoder.get_feature_names_out()

coefs = pd.DataFrame({
    'Feature': column_names,
    'Coefficient': classifier.coef_[0]
}).sort_values(by='Coefficient', ascending=False)

fig = px.bar(
    coefs, 
    x='Coefficient', 
    y='Feature', 
    orientation='h',
    title="Poids des leviers d'action sur la Conversion",
    color='Coefficient',
    color_continuous_scale='RdBu'
)
fig.show()